# 01 — Préparation des données

**Projet :** Segmentation et score d'appétence d'une base clients retail
**Auteure :** Danielle OUATTARA
**Données :** [Customer Personality Analysis — Kaggle](https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis)

**Objectif de ce notebook :** obtenir une base propre et enrichie, réutilisée par tous les notebooks suivants.

| Étape | Contenu |
|---|---|
| 1 | Import et premier contrôle |
| 2 | Valeurs manquantes |
| 3 | Valeurs aberrantes |
| 4 | Recodage des variables qualitatives |
| 5 | Création des variables d'analyse |
| 6 | Contrôles finaux et export |

Mode d'emploi : exécuter les cellules une par une avec **Maj + Entrée**, de haut en bas.

## 1. Import et premier contrôle

In [2]:
import pandas as pd                                   # pandas : manipulation de tableaux de données
import numpy as np                          
# numpy : calculs numériques (utilisé ici pour quelques fonctions)

pd.set_option("display.max_columns", None)            # afficher toutes les colonnes, sans en masquer

df = pd.read_csv("../data/marketing_campaign.csv",    # chemin du fichier, depuis le dossier notebooks/
                 sep="\t")                            # le séparateur de ce fichier est une tabulation, pas une virgule

In [3]:
print(df.shape)                      # (lignes, colonnes) : on attend (2240, 29)
df.head()                                             # aperçu des 5 premières lignes

(2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


In [4]:
df.info()                                             # type de chaque colonne et nombre de valeurs renseignées

<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   str    
 3   Marital_Status       2240 non-null   str    
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   str    
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   int64  
 16 

**Constats attendus :**
- `Income` (revenu) a 2 216 valeurs renseignées sur 2 240 : il manque 24 revenus.
- `Dt_Customer` (date d'entrée client) est lue comme du texte : il faudra la convertir en date.
- `Z_CostContact` et `Z_Revenue` ont la même valeur sur toutes les lignes : ce sont les hypothèses de coût et de revenu de la campagne.

In [5]:
print(df["Z_CostContact"].unique(), df["Z_Revenue"].unique())  # unique() liste les valeurs différentes : on attend [3] et [11]

COUT_CONTACT = int(df["Z_CostContact"].iloc[0])       # coût d'un contact, gardé de côté pour la simulation de ROI
REVENU_REPONSE = int(df["Z_Revenue"].iloc[0])         # revenu d'une réponse positive, gardé de côté
df = df.drop(columns=["Z_CostContact", "Z_Revenue"])  # on retire ces deux colonnes constantes, inutiles pour l'analyse

[3] [11]


In [6]:
print(df["ID"].duplicated().sum())                    # nombre d'identifiants en double : on attend 0

0


## 2. Valeurs manquantes

In [7]:
df.isna().sum()[df.isna().sum() > 0]                  # liste uniquement les colonnes qui ont des valeurs manquantes

Income    24
dtype: int64

**Choix :** remplacer les 24 revenus manquants par le **revenu médian du même niveau d'éducation**.
- La médiane résiste mieux que la moyenne aux revenus extrêmes.
- Le niveau d'éducation est lié au revenu : l'estimation est plus juste qu'une médiane globale.
- Supprimer les lignes ferait perdre 24 clients, et donc une partie de leurs réponses aux campagnes.

In [8]:
mediane_par_education = df.groupby("Education")["Income"].transform("median")  # pour chaque ligne : le revenu médian de son niveau d'éducation
df["Revenu_impute"] = df["Income"].isna().astype(int)                         # 1 si le revenu était manquant (traçabilité du choix)
df["Income"] = df["Income"].fillna(mediane_par_education)                     # remplace uniquement les valeurs manquantes
print(df["Income"].isna().sum())                                              # contrôle : on attend 0

0


## 3. Valeurs aberrantes

In [9]:
df[["Year_Birth", "Income"]].describe()               # min, max, moyenne… pour repérer les valeurs improbables

,Year_Birth,Income
count,2240.000000,2240.000000
mean,1968.805804,52242.594866
std,11.984069,25039.061738
min,1893.000000,1730.000000
25%,1959.000000,35538.750000
50%,1970.000000,51498.500000
75%,1977.000000,68289.750000
max,1996.000000,666666.000000


In [10]:
print(df.loc[df["Year_Birth"] < 1920, ["ID", "Year_Birth"]])   # clients nés avant 1920 : on attend 3 lignes (1893, 1899, 1900)
print(df.loc[df["Income"] > 200000, ["ID", "Income"]])         # revenus extrêmes : on attend une ligne à 666 666

        ID  Year_Birth
192   7829        1900
239  11004        1893
339   1150        1899
        ID    Income
2233  9432  666666.0


**Choix :** supprimer les clients nés avant 1920 (âge supérieur à 94 ans en 2014, probablement des erreurs de saisie) et le revenu de 666 666, isolé et sans rapport avec ses dépenses. Cela représente 4 lignes sur 2 240, soit moins de 0,2 % de la base.

In [11]:
nb_avant = len(df)                                    # nombre de lignes avant suppression
df = df[(df["Year_Birth"] >= 1920)                    # garde les clients nés en 1920 ou après
        & (df["Income"] < 200000)].copy()             # et ceux dont le revenu est inférieur à 200 000 ; copy() crée un tableau indépendant
print(f"{nb_avant - len(df)} lignes supprimées, {len(df)} clients conservés")

4 lignes supprimées, 2236 clients conservés


## 4. Recodage des variables qualitatives

In [12]:
print(df["Marital_Status"].value_counts())            # effectif de chaque situation familiale
print(df["Education"].value_counts())                 # effectif de chaque niveau d'éducation

Marital_Status
Married     864
Together    578
Single      479
Divorced    231
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64
Education
Graduation    1126
PhD            485
Master         370
2n Cycle       201
Basic           54
Name: count, dtype: int64


**Choix pour la situation familiale :** regrouper en deux modalités, ce qui suffit pour décrire le foyer.
- `Married`, `Together` deviennent « En couple ».
- `Single`, `Divorced`, `Widow`, `Alone` deviennent « Seul(e) ».
- `Absurd` et `YOLO` (4 clients) ne sont pas des situations réelles : ces lignes sont supprimées.

**Choix pour l'éducation :** traduire les libellés en français, en regroupant `2n Cycle` (deuxième cycle universitaire) avec `Master`.

In [13]:
df = df[~df["Marital_Status"].isin(["Absurd", "YOLO"])].copy()   # ~ signifie « NON » : on retire ces deux modalités

couple = {"Married": "En couple", "Together": "En couple",       # dictionnaire de correspondance : ancien libellé → nouveau
          "Single": "Seul(e)", "Divorced": "Seul(e)",
          "Widow": "Seul(e)", "Alone": "Seul(e)"}
df["Situation"] = df["Marital_Status"].map(couple)               # map() remplace chaque valeur selon le dictionnaire

education = {"Basic": "Secondaire", "Graduation": "Licence",
             "2n Cycle": "Master", "Master": "Master", "PhD": "Doctorat"}
df["Education_fr"] = df["Education"].map(education)
print(df[["Situation", "Education_fr"]].isna().sum())            # contrôle : 0 valeur non recodée dans chaque colonne

Situation       0
Education_fr    0
dtype: int64


## 5. Création des variables d'analyse

In [14]:
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], dayfirst=True)   # convertit le texte en date ; dayfirst = format jour-mois-année
date_reference = df["Dt_Customer"].max() + pd.Timedelta(days=1)         # date d'analyse = lendemain de la dernière inscription
annee_reference = date_reference.year                                   # année de référence pour calculer l'âge
print(date_reference.date())                                            # on attend 2014-06-30

2014-06-30


In [15]:
df["Age"] = annee_reference - df["Year_Birth"]                                  # âge à la date de l'analyse
df["Anciennete_mois"] = ((date_reference - df["Dt_Customer"]).dt.days / 30.44).round(0)  # ancienneté en mois (30,44 jours en moyenne)
df["Nb_enfants"] = df["Kidhome"] + df["Teenhome"]                               # enfants et adolescents au foyer
df["Taille_foyer"] = df["Nb_enfants"] + np.where(df["Situation"] == "En couple", 2, 1)  # adultes (1 ou 2) + enfants
df["Parent"] = (df["Nb_enfants"] > 0).astype(int)                               # 1 si au moins un enfant au foyer

In [16]:
categories = ["MntWines", "MntFruits", "MntMeatProducts",          # les 6 colonnes de dépenses par catégorie
              "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
df["Depense_totale"] = df[categories].sum(axis=1)                  # somme des 6 catégories, ligne par ligne

canaux = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]  # les 3 canaux d'achat
df["Achats_total"] = df[canaux].sum(axis=1)                        # nombre total d'achats sur les 3 canaux

df["Panier_moyen"] = (df["Depense_totale"]                         # dépense moyenne par achat
                      / df["Achats_total"].replace(0, np.nan))     # replace(0, nan) évite une division par zéro
df["Part_promo"] = (df["NumDealsPurchases"]                        # part des achats faits en promotion
                    / df["Achats_total"].replace(0, np.nan)).clip(upper=1)  # clip(upper=1) plafonne à 100 %

In [17]:
for canal, nom in zip(canaux, ["Part_web", "Part_catalogue", "Part_magasin"]):   # pour chaque canal…
    df[nom] = df[canal] / df["Achats_total"].replace(0, np.nan)                  # …sa part dans les achats du client

for cat in categories:                                                           # pour chaque catégorie de produits…
    df["Part_" + cat.replace("Mnt", "").replace("Products", "").replace("Prods", "")] = (
        df[cat] / df["Depense_totale"].replace(0, np.nan))                       # …sa part dans la dépense du client

In [18]:
campagnes = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5"]
df["Nb_campagnes_acceptees"] = df[campagnes].sum(axis=1)           # combien des 5 premières campagnes le client a acceptées

## 6. Contrôles finaux et export

In [19]:
colonnes_parts = [c for c in df.columns if c.startswith("Part_") or c == "Panier_moyen"]   # colonnes de ratios
print(df[colonnes_parts].isna().sum())                 # clients sans aucun achat : leurs ratios sont vides
df[colonnes_parts] = df[colonnes_parts].fillna(0)      # on les met à 0 : pas d'achat, pas de part

Panier_moyen      6
Part_promo        6
Part_web          6
Part_catalogue    6
Part_magasin      6
Part_Wines        0
Part_Fruits       0
Part_Meat         0
Part_Fish         0
Part_Sweet        0
Part_Gold         0
dtype: int64


In [20]:
df[["Age", "Income", "Anciennete_mois", "Depense_totale",
    "Achats_total", "Panier_moyen", "Part_promo"]].describe().round(1)   # statistiques des nouvelles variables

,Age,Income,Anciennete_mois,Depense_totale,Achats_total,Panier_moyen,Part_promo
count,2232.0,2232.0,2232.0,2232.0,2232.0,2232.0,2232.0
mean,45.1,51942.1,11.6,605.6,12.5,38.1,0.2
std,11.7,21422.0,6.7,602.1,7.2,45.9,0.2
min,18.0,1730.0,0.0,5.0,0.0,0.0,0.0
25%,37.0,35434.8,6.0,69.0,6.0,13.0,0.1
50%,44.0,51445.5,12.0,396.0,12.0,29.6,0.2
75%,55.0,68275.8,17.0,1044.2,18.0,49.1,0.3
max,74.0,162397.0,23.0,2525.0,32.0,1679.0,1.0


In [21]:
print(f"Clients conservés : {len(df)}")
print(f"Taux de réponse à la dernière campagne : {df['Response'].mean():.1%}")   # proportion de 1 dans Response
print(f"Hypothèses ROI : coût d'un contact = {COUT_CONTACT}, revenu par réponse = {REVENU_REPONSE}")

Clients conservés : 2232
Taux de réponse à la dernière campagne : 14.9%
Hypothèses ROI : coût d'un contact = 3, revenu par réponse = 11


In [22]:
df.to_csv("../outputs/clients_prepares.csv", index=False)   # enregistre la base préparée pour les notebooks suivants
print("Export terminé : outputs/clients_prepares.csv")

Export terminé : outputs/clients_prepares.csv


## Synthèse des choix de préparation

| Traitement | Lignes concernées | Justification |
|---|---|---|
| Revenus manquants remplacés par la médiane du niveau d'éducation | 24 | Conserve les clients et leurs réponses ; médiane robuste aux extrêmes |
| Clients nés avant 1920 supprimés | 3 | Âges invraisemblables, erreurs de saisie probables |
| Revenu de 666 666 supprimé | 1 | Valeur isolée, incohérente avec les dépenses |
| Situations `Absurd` et `YOLO` supprimées | 4 | Modalités non exploitables |
| Situation familiale regroupée en 2 modalités | — | Suffisant pour décrire le foyer, évite des groupes trop petits |
| Variables créées | — | Âge, ancienneté, foyer, dépense totale, achats, panier moyen, parts par canal et par catégorie, part promo, nombre de campagnes acceptées |
| Clients sans achat : ratios mis à 0 | 6 | Dépense non nulle mais aucun achat enregistré : incohérence du jeu source, sans impact vu l'effectif |

*Effectifs vérifiés à l'exécution sur le jeu de données complet : 2 232 clients conservés sur 2 240.*